# The Perceptron
**SoC 2026 | Week 2–3**

The perceptron is the fundamental building block of all neural networks. This notebook builds one from scratch, covering:
- Forward pass
- Activation functions
- Binary classification on linearly separable data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Activation Functions

Activation functions introduce non-linearity. Without them, a deep network collapses into a single linear transform.

In [ ]:
def sigmoid(z):  return 1 / (1 + np.exp(-z))
def relu(z):     return np.maximum(0, z)
def tanh(z):     return np.tanh(z)

z = np.linspace(-6, 6, 300)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, fn, name, c in zip(axes,
    [sigmoid, relu, tanh],
    ['Sigmoid σ(z)', 'ReLU max(0,z)', 'Tanh tanh(z)'],
    ['steelblue', 'tomato', 'seagreen']):
    ax.plot(z, fn(z), color=c, lw=2)
    ax.set_title(name, fontsize=11)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.grid(alpha=0.3)

plt.suptitle('Activation Functions', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Binary Classification Dataset

In [ ]:
# Linearly separable 2D data
n = 100
X0 = np.random.randn(n, 2) + np.array([-2, -2])
X1 = np.random.randn(n, 2) + np.array([2, 2])

X = np.vstack([X0, X1])
y = np.hstack([np.zeros(n), np.ones(n)])

plt.figure(figsize=(6, 5))
plt.scatter(*X0.T, label='Class 0', alpha=0.6)
plt.scatter(*X1.T, label='Class 1', alpha=0.6)
plt.legend(); plt.title('Training Data'); plt.show()

## 3. Perceptron from Scratch

In [ ]:
class Perceptron:
    def __init__(self, lr=0.01, epochs=100):
        self.lr     = lr
        self.epochs = epochs
        self.w      = None
        self.b      = None
        self.losses = []

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def _loss(self, y_hat, y):
        # Binary cross-entropy
        eps = 1e-8
        return -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for _ in range(self.epochs):
            z     = X @ self.w + self.b
            y_hat = self._sigmoid(z)

            # Gradients
            dw = (1 / n_samples) * X.T @ (y_hat - y)
            db = (1 / n_samples) * np.sum(y_hat - y)

            self.w -= self.lr * dw
            self.b -= self.lr * db
            self.losses.append(self._loss(y_hat, y))

    def predict(self, X):
        return (self._sigmoid(X @ self.w + self.b) >= 0.5).astype(int)

    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)


model = Perceptron(lr=0.1, epochs=200)
model.fit(X, y)
print(f'Training accuracy: {model.accuracy(X, y) * 100:.2f}%')

plt.plot(model.losses)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Loss (Binary Cross-Entropy)')
plt.grid(alpha=0.3); plt.show()